In [2]:
!pip install duckdb
import duckdb


Defaulting to user installation because normal site-packages is not writeable


In [3]:
import sys
print(sys.executable)

c:\Users\aders\anaconda3\python.exe


In [4]:
import duckdb
import pandas as pd
import os
from datetime import datetime as dt

In [5]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [29]:
arquivo = 'z0019_1.csv'
data_ingestao = dt.now()
df = pd.read_csv(f'../landing/{arquivo}', sep=";")
df['nome_arquivo'] = arquivo
df['data_ingestao'] = data_ingestao
df.head()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-07-20 21:54:57.216765
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-07-20 21:54:57.216765
2,10003,PREGO,BT10,100,500,z0019_1.csv,2026-07-20 21:54:57.216765


In [30]:
con.execute("""
    CREATE TABLE IF NOT EXISTS bronze_produtos (
            NATBR VARCHAR,
            MAKTX VARCHAR,
            WERKS VARCHAR,
            MAINS VARCHAR,
            LABST VARCHAR,
            nome_arquivo VARCHAR,
            data_ingestao TIMESTAMP
    )
""")


In [6]:
resultado = con.execute("SELECT * FROM bronze_z0019").fetchdf()
resultado.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-07-20 21:18:33.655413
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-07-20 21:18:33.655413
2,10003,PREGO,BT10,100,500,z0019_1.csv,2026-07-20 21:18:33.655413
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-07-20 21:18:56.875396
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-07-20 21:18:56.875396
5,10003,PREGO,BT10,100,60,z0019_2.csv,2026-07-20 21:18:56.875396


In [33]:
con.execute("INSERT INTO bronze_produtos SELECT * FROM df")

In [42]:
arquivo = 'z0019_2.csv'
data_ingestao = dt.now()
df = pd.read_csv(f'../landing/{arquivo}', sep=";")
df['nome_arquivo'] = arquivo
df['data_ingestao'] = data_ingestao
df.head()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10004,SERRA,BT50,100,200,z0019_2.csv,2026-07-20 22:29:05.788667
1,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-07-20 22:29:05.788667
2,10003,PREGO,BT10,100,60,z0019_2.csv,2026-07-20 22:29:05.788667


In [13]:
con.execute("ALTER TABLE bronze_produtos RENAME TO bronze_z0019")

In [23]:
con.close()


In [10]:
resultado.head(10)


,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-07-20 21:18:33.655413
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-07-20 21:18:33.655413
2,10003,PREGO,BT10,100,500,z0019_1.csv,2026-07-20 21:18:33.655413
3,10004,SERRA,BT50,100,200,z0019_2.csv,2026-07-20 21:18:56.875396
4,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-07-20 21:18:56.875396
5,10003,PREGO,BT10,100,60,z0019_2.csv,2026-07-20 21:18:56.875396


In [8]:
df = con.execute("""
                 select * from (
                    SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row_num
                    FROM bronze_z0019
                    WHERE data_ingestao >= '2026-07-01 00:00:00' 
                ) where row_num = 1
                """).fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row_num
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-07-20 21:18:33.655413,1
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-07-20 21:18:33.655413,1
2,10004,SERRA,BT50,100,200,z0019_2.csv,2026-07-20 21:18:56.875396,1
3,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-07-20 21:18:56.875396,1
4,10003,PREGO,BT10,100,60,z0019_2.csv,2026-07-20 21:18:56.875396,1


In [11]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row_num'])
df_final.head(6)

,NATBR,MAKTX,WERKS,MAINS,LABST
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,SERRA,BT50,100,200
3,10005,MACHADO,BT50,100,100
4,10003,PREGO,BT10,100,60


In [12]:
df_final = df_final.rename(columns={
    'NATBR': 'id',
    'MAKTX': 'nm_produto',
    'WERKS': 'id_categoria',
    'MAINS': 'id_fornecedor',
    'LABST': 'vl_preco'}
)

In [13]:
df_final.head(6)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,SERRA,BT50,100,200
3,10005,MACHADO,BT50,100,100
4,10003,PREGO,BT10,100,60


In [ ]:
df2 = df_final.astype({
df2.dtypes


id                        int64
nm_produto       string[python]
id_categoria     string[python]
id_fornecedor             int64
vl_preco                float64
dtype: object

In [22]:
df2 = df2.astype({
    'id': int,
    'nm_produto': str,
    'id_categoria': str,
    'id_fornecedor': int,
    'vl_preco': float
})
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [23]:
df2.dtypes


id                 int64
nm_produto        object
id_categoria      object
id_fornecedor      int64
vl_preco         float64
dtype: object

In [24]:
con.execute("""
    CREATE TABLE IF NOT EXISTS produtos (
        id BIGINT PRIMARY KEY,
        nm_produto TEXT,
        id_categoria TEXT,
        id_fornecedor BIGINT,
        vl_preco REAL
    )
""")

In [26]:
con.execute("INSERT INTO produtos SELECT * FROM df2")

In [27]:
df_resultado = con.execute("SELECT * FROM produtos").fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [28]:
con.close()